# Chapter 9 §9.5: Fashion Video Generator

The evaluator-optimizer pattern scores generated images before the downstream video call, where failures are substantially more expensive.


In [ ]:
%pip install -r ../requirements.txt -q


## Runtime setup

Cells tagged `chapter-example` contain the chapter's core examples. Supporting cells provide imports, fixtures, or safe local stand-ins for external services.


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

import dspy

env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY in ../.env before running this notebook.")

lm = dspy.LM("openai/gpt-4o-mini", api_key=api_key)
dspy.configure(lm=lm)


## External generation stand-ins

The image stand-in returns a tiny valid PNG data URL so `dspy.Image` can validate it without a paid image-generation call.


In [ ]:
import hashlib

_ONE_PIXEL_PNG = "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAABAAAAAQCAIAAACQkWg2AAAAI0lEQVR4nGNsaGhgIAUwkaSaYVQDcYCJSHVwMKqBGEByKAEAyUwBoHKcrY0AAAAASUVORK5CYII="

def generate_image(prompt: str, style: str = "photorealistic") -> str:
    del prompt, style
    return _ONE_PIXEL_PNG

def generate_video(image_url: str, motion: str) -> str:
    digest = hashlib.sha256((image_url + motion).encode()).hexdigest()[:12]
    return f"https://placeholder.example/video-{digest}.mp4"


## Text-to-image module


In [ ]:
class FashionPrompt(dspy.Signature):
    """Generate detailed image prompt for fashion video."""

    concept = dspy.InputField(
        desc="fashion concept (e.g., 'sustainable streetwear')"
    )
    target_audience = dspy.InputField(desc="target demographic")

    image_prompt = dspy.OutputField(
        desc="detailed Flux prompt for photorealistic image"
    )
    style_notes = dspy.OutputField(desc="visual style guidance")


class ImageGenerator(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate_prompt = dspy.ChainOfThought(FashionPrompt)

    def forward(self, concept, target_audience):
        prompt_result = self.generate_prompt(
            concept=concept,
            target_audience=target_audience
        )

        # Call external image generation API
        image_url = generate_image(
            prompt_result.image_prompt,
            prompt_result.style_notes
        )

        return dspy.Prediction(
            image_url=image_url,
            image_prompt=prompt_result.image_prompt,
            style_notes=prompt_result.style_notes
        )


## Image-to-video module


In [ ]:
class VideoGenerator(dspy.Module):
    def __init__(self):
        super().__init__()
        self.image_gen = ImageGenerator()

    def forward(self, concept, target_audience,
                motion="smooth camera pan, model walks"):
        # Stage 1: Generate image
        image_result = self.image_gen(
            concept=concept,
            target_audience=target_audience
        )

        # Stage 2: Generate video from image
        video_url = generate_video(
            image_result.image_url, motion
        )

        return dspy.Prediction(
            video_url=video_url,
            image_url=image_result.image_url,
            image_prompt=image_result.image_prompt,
            style_notes=image_result.style_notes
        )


## Vision-language evaluator


In [ ]:
class ImageQuality(dspy.Signature):
    """Evaluate whether a generated image matches the fashion concept."""

    concept = dspy.InputField()
    image: dspy.Image = dspy.InputField()

    matches_concept = dspy.OutputField(
        desc="1-10: how well image matches concept"
    )
    visual_appeal = dspy.OutputField(desc="1-10: aesthetic quality")
    overall_score = dspy.OutputField(desc="1-10: overall rating")


image_evaluator = dspy.ChainOfThought(ImageQuality)


def image_reward(args, pred):
    result = image_evaluator(
        concept=args["concept"],
        image=dspy.Image(pred.image_url)
    )
    return float(result.overall_score) / 10


## Refine wrapper


In [ ]:
refined_generator = dspy.Refine(
    module=ImageGenerator(),
    N=3,
    reward_fn=image_reward,
    threshold=0.8,
)
